# Sparse Autoencoders for LLM Interpretability

In notebooks 01 and 02, we explored circuits and superposition. We saw that neural networks pack way more features than dimensions into their activation spaces. The natural next question: **how do we get those features back out?**

## Section 1: From Superposition to Dictionary Learning

We know features are directions in superposition. How do we find them? **Sparse Autoencoders (SAEs)** learn a dictionary of feature directions by training an autoencoder with a sparsity constraint.

Here's the setup:
- Input: activation vector $x \in \mathbb{R}^d$ from a specific layer of the LLM
- Encoder: $f(x) = \text{ReLU}(W_{\text{enc}} \cdot (x - b_{\text{dec}}) + b_{\text{enc}})$, where $f(x) \in \mathbb{R}^n$, $n \gg d$
- Decoder: $\hat{x} = W_{\text{dec}} \cdot f(x) + b_{\text{dec}}$
- Loss: $\mathcal{L} = \|x - \hat{x}\|^2 + \lambda \cdot \|f(x)\|_1$

The L1 penalty encourages sparsity -- only a few features active for any given input. Each column of $W_{\text{dec}}$ represents a learned feature direction. The model learns $n \gg d$ features while keeping activations sparse.

Key design choices:
- **Expansion factor**: $n/d$ ratio (typically 4x-256x). Larger = more features but harder to train
- **L1 coefficient** $\lambda$: controls sparsity vs reconstruction quality
- **TopK variant**: Instead of L1, keep only top-k activations ([Gao et al., 2024](https://arxiv.org/abs/2406.04093))

### SAE Loss Landscape and Training Dynamics

The SAE objective encodes a fundamental tension between faithfulness and parsimony. Understanding its structure is essential for effective training.

**The full loss.** For input activations $x \in \mathbb{R}^d$, the SAE loss is:

$$\mathcal{L} = \underbrace{\mathbb{E}_x\!\left[\left\|x - \left(W_{\text{dec}} \cdot \text{ReLU}\!\left(W_{\text{enc}}(x - b_{\text{dec}}) + b_{\text{enc}}\right) + b_{\text{dec}}\right)\right\|^2\right]}_{\text{reconstruction}} + \;\lambda \cdot \underbrace{\mathbb{E}_x\!\left[\left\|f(x)\right\|_1\right]}_{\text{sparsity}}$$

where $f(x) = \text{ReLU}(W_{\text{enc}}(x - b_{\text{dec}}) + b_{\text{enc}})$ is the vector of feature activations.

**The L1-reconstruction tension.** These two terms push in opposite directions:

- **Reconstruction loss** is minimized when $f(x)$ captures maximum information about $x$ — this favors *many* active features with large activations.
- **L1 penalty** is minimized when $f(x) = 0$ — no features active at all.

The balance point depends on $\lambda$. For a single feature $i$, the equilibrium condition (setting the gradient to zero) gives a soft-thresholding behavior: feature $i$ activates only when the pre-ReLU input exceeds a threshold that scales with $\lambda / \|w_{\text{dec}}^i\|$.

**The Pareto frontier.** Varying $\lambda$ traces a curve in (reconstruction error, L0) space:
- $\lambda \to 0$: many active features, low reconstruction error, but features are not sparse and hard to interpret.
- $\lambda \to \infty$: all features inactive, reconstruction error equals $\text{Var}(x)$.
- Optimal $\lambda$: depends on the application. For interpretability, we want L0 $\ll d_{\text{model}}$.

**Dead neurons: a critical failure mode.** A feature $i$ is "dead" if $b_{\text{enc}}^i$ becomes sufficiently negative that $W_{\text{enc}}^i \cdot (x - b_{\text{dec}}) + b_{\text{enc}}^i < 0$ for all $x$ in the data distribution. Once dead:
1. The feature activation is always zero (due to ReLU).
2. The gradient $\partial \mathcal{L} / \partial W_{\text{enc}}^i = 0$ (ReLU gate is closed).
3. The gradient $\partial \mathcal{L} / \partial b_{\text{enc}}^i = 0$ (same reason).
4. The feature can **never recover** through gradient descent alone — it is a fixed point of training.

This is not merely a capacity issue: dead features represent wasted parameters. With expansion factors of 64x or more, 50-90% of features may die without intervention.

**Solution 1: Neuron resampling.** Periodically identify dead features (those with zero activation over a large batch) and reinitialize them:
- Set $W_{\text{enc}}^i$ to the direction of a high-loss input example (normalized).
- Set $b_{\text{enc}}^i$ to a small positive value (ensuring initial activation).
- Set $W_{\text{dec}}^i$ to a small-norm vector in the same direction.

This gives the feature a "second chance" in a part of activation space that the SAE currently reconstructs poorly.

**Solution 2: Ghost gradients (simplified).** Instead of discrete resampling, provide a continuous synthetic gradient signal to dead neurons. When feature $i$ is dead, replace its zero gradient with a gradient pointing toward high-reconstruction-error inputs. This keeps dead neurons "warm" and allows them to drift back to useful directions.

**The TopK alternative (Gao et al. 2024).** Replace the L1 penalty entirely: instead of soft thresholding via L1, keep exactly the top-$k$ activations and zero out the rest:

$$f_{\text{TopK}}(x) = \text{TopK}\!\left(W_{\text{enc}}(x - b_{\text{dec}}) + b_{\text{enc}},\; k\right)$$

The loss becomes simply $\mathcal{L} = \mathbb{E}_x[\|x - \hat{x}\|^2]$ (no L1 term). This has two advantages: (1) **exact sparsity control** — L0 $= k$ by construction, eliminating the need to tune $\lambda$; (2) **no shrinkage** — L1 biases all activations toward zero (a well-known issue in LASSO regression), while TopK preserves activation magnitudes. The tradeoff: TopK enforces the *same* sparsity for every input, whereas L1 allows variable sparsity (some inputs may genuinely need more features than others).

## Section 2: Training an SAE from Scratch

Let's build a basic SAE and train it on GPT-2 activations. We'll use residual stream activations from a middle layer.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

class SparseAutoencoder(nn.Module):
    """Sparse Autoencoder for extracting features from LLM activations."""
    def __init__(self, d_model, n_features, l1_coeff=1e-3):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        # Note: l1_coeff is stored here for reference/logging but is applied
        # externally in the training loop (see Section 2) to keep the loss
        # computation explicit and flexible for experimentation.
        self.l1_coeff = l1_coeff
        
        # Encoder
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) * (1 / np.sqrt(d_model)))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder (tied to encoder direction by convention, but separate parameter)
        self.W_dec = nn.Parameter(torch.randn(n_features, d_model) * (1 / np.sqrt(n_features)))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        
        # Normalize decoder weights to unit norm
        with torch.no_grad():
            self.W_dec.data = self.W_dec.data / self.W_dec.data.norm(dim=-1, keepdim=True)
    
    def encode(self, x):
        """Encode activations to sparse feature activations."""
        return torch.relu((x - self.b_dec) @ self.W_enc + self.b_enc)
    
    def decode(self, f):
        """Decode feature activations back to model space."""
        return f @ self.W_dec + self.b_dec
    
    def forward(self, x):
        f = self.encode(x)
        x_hat = self.decode(f)
        
        # Losses
        reconstruction_loss = (x - x_hat).pow(2).sum(dim=-1).mean()
        sparsity_loss = f.abs().sum(dim=-1).mean()
        
        return x_hat, f, reconstruction_loss, sparsity_loss
    
    @torch.no_grad()
    def normalize_decoder(self):
        """Keep decoder columns at unit norm (important for training stability)."""
        self.W_dec.data = self.W_dec.data / self.W_dec.data.norm(dim=-1, keepdim=True)

print("SAE architecture defined.")

### Why Unit Norm Decoder Columns?

The constraint $\|w_{\text{dec}}^i\| = 1$ for all feature directions $i$ is not merely a regularization choice — it is necessary for the loss to be well-defined.

**The scaling degeneracy.** Without normalization, the SAE can exploit a trivial optimization:

$$W_{\text{enc}}^i \leftarrow \alpha \, W_{\text{enc}}^i, \quad W_{\text{dec}}^i \leftarrow \frac{1}{\alpha}\, W_{\text{dec}}^i, \quad b_{\text{enc}}^i \leftarrow \alpha \, b_{\text{enc}}^i$$

For any $\alpha > 1$: the reconstruction $\hat{x}$ is unchanged (the $\alpha$ factors cancel), but the feature activation $f_i(x) = \text{ReLU}(\alpha \cdot (\cdots))$ is scaled by $\alpha$, which means $\|f(x)\|_1$ is scaled by $\alpha$ as well — **but the encoder can compensate by also scaling $b_{\text{enc}}$** to keep the same set of active features while shrinking their magnitudes. In the other direction, setting $\alpha \ll 1$ shrinks the encoder and inflates the decoder, reducing $\|f(x)\|_1$ without changing the reconstruction quality. The L1 loss goes to zero while the model learns nothing useful.

**The fix.** Constraining $\|w_{\text{dec}}^i\| = 1$ breaks this degeneracy. Now:
- The feature activation $f_i(x)$ directly represents the **magnitude** of feature $i$'s contribution to the residual stream, since $\hat{x} = \sum_i f_i(x) \, w_{\text{dec}}^i + b_{\text{dec}}$ and $\|w_{\text{dec}}^i\| = 1$.
- The L1 penalty on $f_i(x)$ has a consistent interpretation: it penalizes the total magnitude of feature contributions, not an arbitrary rescaling thereof.

**Decoder columns as feature directions.** With unit norm, the columns of $W_{\text{dec}}$ are literally the feature directions in $\mathbb{R}^{d_{\text{model}}}$. The cosine similarity between two decoder columns $\langle w_{\text{dec}}^i, w_{\text{dec}}^j \rangle$ is their geometric interference — exactly the quantity that appears in the superposition theory from Notebook 02.

**Encoder-decoder relationship.** In a linear autoencoder without sparsity ($\lambda = 0$), the optimal solution satisfies $W_{\text{enc}} = W_{\text{dec}}^\top$ (the encoder is the transpose of the decoder). Note: in practice this relationship only holds at initialization or in the linear/no-sparsity limit; after training with L1 sparsity, the encoder and decoder weights diverge significantly. With L1 sparsity, this symmetry breaks:
- The encoder must implement a **thresholding** operation: features should only activate when the input has a strong enough component in the feature direction.
- The decoder must implement **reconstruction**: mapping sparse feature activations back to the full $d_{\text{model}}$-dimensional space.
- Empirically, $W_{\text{enc}} \approx W_{\text{dec}}^\top$ holds approximately, but the encoder rows are biased toward *detecting* features (sharpened, with thresholds set by $b_{\text{enc}}$) while decoder columns represent the *contribution* of each feature to the residual stream. The L1 penalty forces the encoder to be more selective than the decoder is expressive.

In [ ]:
from transformer_lens import HookedTransformer

# Load model
model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

# We'll collect residual stream activations from layer 6 (middle of the network)
hook_point = "blocks.6.hook_resid_post"

# Collect activations from a simple dataset
texts = [
    "The president of the United States lives in the White House.",
    "Machine learning models are trained using gradient descent.",
    "The capital of France is Paris, which is known for the Eiffel Tower.",
    "Python is a popular programming language used in data science.",
    "The stock market experienced significant volatility today.",
    "Quantum computing could revolutionize cryptography and drug discovery.",
    "The Great Wall of China is visible from certain low Earth orbits.",  # Note: this is a common misconception used here as example training data; the Wall is not readily visible from space with the naked eye.
    "Neural networks loosely mimic biological brain structure.",
    "Climate change affects weather patterns across the globe.",
    "The speed of light in a vacuum is approximately 300,000 km per second.",
] * 20  # Repeat for more data

all_activations = []
for text in texts:
    _, cache = model.run_with_cache(text, names_filter=hook_point)
    acts = cache[hook_point][0]  # (seq_len, d_model)
    all_activations.append(acts.detach().cpu())

activations = torch.cat(all_activations, dim=0)  # (total_tokens, d_model)
print(f"Collected {activations.shape[0]} activation vectors of dimension {activations.shape[1]}")

In [ ]:
# SAE hyperparameters
n_features = 768 * 4  # 4x expansion
l1_coeff = 3e-4
lr = 1e-3
n_epochs = 50
batch_size = 256

sae = SparseAutoencoder(d_model=768, n_features=n_features, l1_coeff=l1_coeff).to(device)
optimizer = optim.Adam(sae.parameters(), lr=lr)
activations_device = activations.to(device)

history = {"reconstruction": [], "sparsity": [], "total": [], "l0": []}

for epoch in tqdm(range(n_epochs)):
    # Shuffle
    perm = torch.randperm(activations_device.shape[0])
    epoch_losses = {"reconstruction": 0, "sparsity": 0, "total": 0, "l0": 0}
    n_batches = 0
    
    for i in range(0, len(perm), batch_size):
        batch = activations_device[perm[i:i+batch_size]]
        x_hat, f, recon_loss, sparse_loss = sae(batch)
        
        loss = recon_loss + l1_coeff * sparse_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        sae.normalize_decoder()
        
        epoch_losses["reconstruction"] += recon_loss.item()
        epoch_losses["sparsity"] += sparse_loss.item()
        epoch_losses["total"] += loss.item()
        epoch_losses["l0"] += (f > 0).float().sum(dim=-1).mean().item()
        n_batches += 1
    
    for k in epoch_losses:
        history[k].append(epoch_losses[k] / n_batches)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: recon={history['reconstruction'][-1]:.4f}, "
              f"L0={history['l0'][-1]:.1f}/{n_features}")

print(f"\nFinal: recon={history['reconstruction'][-1]:.4f}, L0={history['l0'][-1]:.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["reconstruction"])
axes[0].set_title("Reconstruction Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["sparsity"])
axes[1].set_title("Sparsity Loss (L1)")
axes[1].set_xlabel("Epoch")

axes[2].plot(history["l0"])
axes[2].set_title("L0 (avg active features)")
axes[2].set_xlabel("Epoch")
axes[2].axhline(y=768, color='r', linestyle='--', alpha=0.5, label='d_model')
axes[2].legend()

plt.suptitle("SAE Training Progress", fontsize=14)
plt.tight_layout()
plt.show()

## Section 3: Analyzing Learned Features

Now let's look at what the SAE actually learned. We can:
1. Find the most active features for a given input
2. Look at the decoder directions (what each feature "means" in model space)
3. Check for dead features (features that never activate)

In [ ]:
# Run all activations through the SAE
with torch.no_grad():
    all_features = sae.encode(activations_device)

# Feature activation statistics
feature_freq = (all_features > 0).float().mean(dim=0).cpu().numpy()
feature_mean_act = all_features.mean(dim=0).cpu().numpy()

# Dead features (never activate)
n_dead = (feature_freq == 0).sum()
print(f"Dead features: {n_dead}/{n_features} ({100*n_dead/n_features:.1f}%)")
print(f"Active features: {n_features - n_dead}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Activation frequency distribution
axes[0].hist(feature_freq[feature_freq > 0], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel("Activation Frequency")
axes[0].set_ylabel("Count")
axes[0].set_title("Feature Activation Frequency Distribution")
axes[0].set_yscale("log")

# Mean activation distribution
axes[1].hist(feature_mean_act[feature_mean_act > 0], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel("Mean Activation")
axes[1].set_ylabel("Count")
axes[1].set_title("Feature Mean Activation Distribution")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

In [ ]:
# Encode a specific sentence and find its top features
test_text = "The president of the United States lives in the White House."
_, cache = model.run_with_cache(test_text, names_filter=hook_point)
test_acts = cache[hook_point][0].to(device)
test_tokens = [model.tokenizer.decode(t) for t in model.to_tokens(test_text)[0]]

with torch.no_grad():
    test_features = sae.encode(test_acts)

# For each token position, show top-k active features
k = 5
print(f"Top {k} features per token position:\n")
for pos in range(len(test_tokens)):
    top_feats = test_features[pos].topk(k)
    feat_str = ", ".join([f"f{idx.item()}({val.item():.2f})" 
                          for val, idx in zip(top_feats.values, top_feats.indices)])
    print(f"  '{test_tokens[pos]}' -> {feat_str}")

## Section 4: Using SAELens (Production Tool)

In practice, you'll want to use **SAELens** for training and analyzing SAEs. It handles:
- Efficient activation collection from large models
- Dead neuron resampling
- Decoder norm constraints
- Integration with Neuronpedia for visualization
- Loading pretrained SAEs from the community

In [ ]:
# NOTE: This requires sae_lens to be installed and may need GPU
# pip install sae_lens

try:
    from sae_lens import SAE
    
    # Load a pretrained SAE for GPT-2 small from the SAELens library
    # These are trained on much more data than our toy example above
    sae_lens_model, cfg_dict, sparsity = SAE.from_pretrained(
        release="gpt2-small-res-jb",  # Joseph Bloom's GPT-2 SAEs
        sae_id="blocks.6.hook_resid_post",  # Same layer we used
    )
    print(f"Loaded pretrained SAE:")
    print(f"  Features: {sae_lens_model.cfg.d_sae}")
    print(f"  Input dim: {sae_lens_model.cfg.d_in}")
    print("  Ready for analysis!")
    
except ImportError:
    print("sae_lens not installed. Install with: pip install sae_lens")
    print("Pretrained SAEs provide much better feature quality than our toy training above.")
except Exception as e:
    print(f"Could not load pretrained SAE: {e}")
    print("This may require downloading model weights. See SAELens docs for details.")

## Section 5: SAE Evaluation -- How Good Are the Features?

Evaluating SAE quality involves several metrics:

1. **Reconstruction fidelity**: MSE between original and reconstructed activations. Also measured as fraction of variance explained (FVE).

2. **Downstream loss**: Replace the model's activations with SAE reconstructions and measure the increase in language modeling loss. Small increase = good reconstruction.

3. **Sparsity (L0)**: Average number of active features per input. Lower = sparser = more interpretable.

4. **Interpretability score**: Have humans (or an LM judge) rate whether each feature has a clear, consistent meaning. The "Towards Monosemanticity" paper found ~70% of features were interpretable. (Note: the exact figure varies by methodology, SAE configuration, and model; this is an approximate summary rather than a single definitive number.)

5. **Dead features**: Features that never activate are wasted capacity. Good training keeps this low.

The fundamental tradeoff: **reconstruction quality vs sparsity**. More sparsity leads to cleaner features but worse reconstruction. The Pareto frontier of this tradeoff is what we optimize.

In [ ]:
with torch.no_grad():
    # Reconstruction quality
    x_hat, f, recon_loss, _ = sae(activations_device)
    
    # Fraction of variance explained
    total_var = activations_device.var(dim=0).sum()
    residual_var = (activations_device - x_hat).var(dim=0).sum()
    fve = 1 - residual_var / total_var
    
    # L0 sparsity
    l0 = (f > 0).float().sum(dim=-1).mean()
    
    # Dead features
    feature_active = (f > 0).any(dim=0)
    n_alive = feature_active.sum()

print("SAE Evaluation Metrics:")
print(f"  Reconstruction MSE: {recon_loss.item():.4f}")
print(f"  Fraction of Variance Explained: {fve.item():.4f}")
print(f"  L0 (avg active features): {l0.item():.1f} / {n_features}")
print(f"  L0 / d_model: {l0.item() / 768:.2f}")
print(f"  Alive features: {n_alive.item()} / {n_features}")
print(f"  Dead features: {n_features - n_alive.item()}")

---
### Running Example: IOI — Extracting IOI Features

This is part of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

**The task**: "When Mary and John went to the store, John gave a drink to" — the model should predict "Mary".

Here we run the IOI prompt through the SAE trained earlier and examine which features activate most strongly at the final token position — the position where the model decides to output "Mary". These features represent the interpretable concepts the model is using at the moment of prediction.

In [ ]:
# Running Example: IOI — SAE features at the prediction position
prompt = "When Mary and John went to the store, John gave a drink to"
_, cache = model.run_with_cache(prompt, names_filter="blocks.6.hook_resid_post")
ioi_acts = cache["blocks.6.hook_resid_post"][0, -1:].to(device)

with torch.no_grad():
    ioi_features = sae.encode(ioi_acts)

top_feats = ioi_features[0].topk(10)
print(f"Prompt: '{prompt}'")
print(f"\nTop 10 SAE features at the final (prediction) position:")
for val, idx in zip(top_feats.values, top_feats.indices):
    print(f"  Feature {idx.item()}: {val.item():.3f}")

print(f"\nTotal active features at this position: {(ioi_features[0] > 0).sum().item()}")
print("These features represent the interpretable concepts the model uses when deciding to output 'Mary'.")

---
## Exercises

### Exercise 1: L1 Penalty Sweep

Train multiple small SAEs with different L1 coefficients (1e-4, 1e-3, 1e-2, 1e-1). For each, measure:
1. Reconstruction MSE
2. Average L0 (number of active features per input)
3. Percentage of dead features after training

Plot the Pareto frontier of reconstruction loss vs. sparsity (L0). Which L1 coefficient gives the best tradeoff?

<details>
<summary>Hint</summary>

Reuse the `SparseAutoencoder` class defined earlier in this notebook with different `l1_coeff` values. After training each SAE, compute: MSE via `(x - x_hat).pow(2).sum(-1).mean()`, L0 via `(f > 0).float().sum(-1).mean()`, and dead features as features with zero activation frequency over the full dataset. Use `plt.scatter()` for the Pareto plot with L0 on x-axis and MSE on y-axis.

</details>

In [ ]:
l1_values = [1e-4, 1e-3, 1e-2, 1e-1]
results = []

# TODO: Try adding more L1 values or changing n_epochs!
for l1 in l1_values:
    print(f"\nTraining SAE with L1={l1}...")
    sae_sweep = SparseAutoencoder(d_model=768, n_features=768*4, l1_coeff=l1).to(device)
    optimizer_sweep = optim.Adam(sae_sweep.parameters(), lr=1e-3)
    
    for epoch in range(30):
        perm = torch.randperm(activations_device.shape[0])
        for i in range(0, len(perm), 256):
            batch = activations_device[perm[i:i+256]]
            x_hat, f, recon_loss, sparse_loss = sae_sweep(batch)
            loss = recon_loss + l1 * sparse_loss
            optimizer_sweep.zero_grad()
            loss.backward()
            optimizer_sweep.step()
            sae_sweep.normalize_decoder()
    
    # Evaluate
    with torch.no_grad():
        x_hat, f, mse, _ = sae_sweep(activations_device)
        l0 = (f > 0).float().sum(-1).mean().item()
        dead_pct = ((f > 0).float().mean(0) == 0).float().mean().item() * 100
    
    results.append({'l1': l1, 'mse': mse.item(), 'l0': l0, 'dead_pct': dead_pct})
    print(f"  MSE={mse.item():.4f}, L0={l0:.1f}, Dead={dead_pct:.1f}%")

# Plot Pareto frontier
fig, ax = plt.subplots(figsize=(8, 6))
for r in results:
    ax.scatter(r['l0'], r['mse'], s=100, zorder=5)
    ax.annotate(f"L1={r['l1']}", (r['l0'], r['mse']), textcoords="offset points", xytext=(10, 5))
ax.set_xlabel("L0 (avg active features)")
ax.set_ylabel("Reconstruction MSE")
ax.set_title("Pareto Frontier: Reconstruction vs. Sparsity")
ax.grid(True, alpha=0.3)
plt.show()

### Exercise 2: Manual SAE Forward Pass

Implement a single SAE forward pass from scratch (no library). Given encoder weights W_enc, decoder W_dec, biases b_enc, b_dec from the trained SAE above, manually compute:
1. `z = ReLU(W_enc @ (x - b_dec) + b_enc)` (feature activations)
2. `x_hat = W_dec @ z + b_dec` (reconstruction)

Compare your manual output with the SAE's built-in forward pass to verify they match.

<details>
<summary>Hint</summary>

Extract weights from the trained SAE using `sae.W_enc.data`, `sae.W_dec.data`, `sae.b_enc.data`, `sae.b_dec.data`. For a test input `x` (a single activation vector), the manual computation is: `z = torch.relu((x - b_dec) @ W_enc + b_enc)` and `x_hat = z @ W_dec + b_dec`. Compare using `torch.allclose(manual_x_hat, sae_x_hat, atol=1e-5)`.

</details>

In [ ]:
# Exercise 2: Manual SAE Forward Pass

# Get a test activation vector
x = activations_device[0:1]  # shape: (1, 768)

# 1. Extract weights from the trained SAE
W_enc = sae.W_enc.data   # (d_model, n_features) = (768, 3072)
W_dec = sae.W_dec.data   # (n_features, d_model) = (3072, 768)
b_enc = sae.b_enc.data   # (n_features,) = (3072,)
b_dec = sae.b_dec.data   # (d_model,) = (768,)

# 2. Manually compute the forward pass
# TODO: Try stepping through each line to understand the data flow!
z = torch.relu((x - b_dec) @ W_enc + b_enc)    # feature activations
x_hat_manual = z @ W_dec + b_dec                 # reconstruction

# 3. Compare with the SAE's built-in forward pass
with torch.no_grad():
    x_hat_sae, f_sae, _, _ = sae(x)

# 4. Verify they match
print(f"Manual z shape: {z.shape}")
print(f"SAE f shape: {f_sae.shape}")
print(f"Feature activations match: {torch.allclose(z, f_sae, atol=1e-5)}")
print(f"Reconstructions match: {torch.allclose(x_hat_manual, x_hat_sae, atol=1e-5)}")
print(f"Max difference (features): {(z - f_sae).abs().max().item():.2e}")
print(f"Max difference (reconstruction): {(x_hat_manual - x_hat_sae).abs().max().item():.2e}")

## Section 6: Key Takeaways & Further Reading

**What you should remember:**
- SAEs decompose polysemantic activations into sparse, monosemantic features
- The L1 penalty (or TopK) enforces sparsity -- each input activates few features
- Feature quality depends on expansion factor, sparsity, training data, and compute
- Pretrained SAEs (via SAELens/Gemma Scope) are much better than toy examples
- The reconstruction-sparsity Pareto frontier is the key optimization target

**The SAE pipeline in practice:**
1. Choose layer and activation type (residual stream, MLP output, attention output)
2. Collect activations from diverse text
3. Train SAE with appropriate hyperparameters
4. Evaluate: reconstruction, sparsity, interpretability
5. Use features for downstream analysis (circuits, steering, monitoring)

**Further reading:**
- [Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features) (Anthropic, 2023) -- the seminal paper
- [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) (Anthropic, 2024) -- scaling to Claude 3 Sonnet
- [Scaling and Evaluating SAEs](https://arxiv.org/abs/2406.04093) (Gao et al., 2024) -- TopK SAEs, evaluation methodology
- [SAELens documentation](https://github.com/jbloom/SAELens)
- [Neuronpedia](https://www.neuronpedia.org/) -- browse trained features interactively

**Next**: Notebook 04 -- Activation Patching (using causal methods to find circuits)